# 🏛️ KG Hà Nội — Bước 2 + 2.5: Extract Triplets & Markdown

**Input (từ bước 1 đã chạy):**
- `wiki_raw/` — bài viết Wikipedia
- `food_raw/` — dữ liệu ẩm thực

**Output:**
- `triplets_raw/` — triplets di tích (LLM extract)
- `triplets_food/` — triplets ẩm thực (normalized)
- `kg_markdown/` — Markdown files cho human review

## 0. Setup & Kiểm tra input

In [2]:
!pip install transformers accelerate sentencepiece -q

import json, re, os
from pathlib import Path
from collections import defaultdict

# ---- SỬA ĐƯỜNG DẪN NẾU CẦN ----
WIKI_RAW     = "/kaggle/input/datasets/thinbotng/kgscrape/wiki_raw"
FOOD_RAW     = "/kaggle/input/datasets/thinbotng/kgscrape/food_raw"
OUT_TRIPLETS = "/kaggle/working/triplets_raw"
OUT_FOOD     = "/kaggle/working/triplets_food"
OUT_MD       = "/kaggle/working/kg_markdown"

os.makedirs(OUT_TRIPLETS, exist_ok=True)
os.makedirs(OUT_FOOD, exist_ok=True)
os.makedirs(OUT_MD, exist_ok=True)

# Kiểm tra input
n_wiki = len(list(Path(WIKI_RAW).glob("*.json"))) if Path(WIKI_RAW).exists() else 0
n_food_r = len(list((Path(FOOD_RAW)/"restaurants").glob("*.json"))) if (Path(FOOD_RAW)/"restaurants").exists() else 0
n_food_d = len(list((Path(FOOD_RAW)/"dishes").glob("*.json"))) if (Path(FOOD_RAW)/"dishes").exists() else 0
n_food_l = len(list((Path(FOOD_RAW)/"districts").glob("*.json"))) if (Path(FOOD_RAW)/"districts").exists() else 0

print(f"Wiki articles:  {n_wiki}")
print(f"Restaurants:    {n_food_r}")
print(f"Dishes:         {n_food_d}")
print(f"Districts:      {n_food_l}")

assert n_wiki > 0, f"Không tìm thấy wiki data ở {WIKI_RAW}!"
print("\n✓ Input OK")

Wiki articles:  91
Restaurants:    20
Dishes:         17
Districts:      16

✓ Input OK


## Bước 2a: Normalization maps + Prompt

Chạy cell này để define các hàm normalize, validate, và prompt cho LLM.

In [3]:
# ---- Relation normalization ----
RELATION_NORMALIZE = {
    "năm_xây_dựng": "xây_dựng_năm", "xây_dựng": "xây_dựng_năm",
    "xây_năm": "xây_dựng_năm", "năm_xây": "xây_dựng_năm",
    "khởi_công": "xây_dựng_năm", "hoàn_thành": "xây_dựng_năm",
    "hoàn_thành_năm": "xây_dựng_năm", "năm_hoàn_thành": "xây_dựng_năm",
    "được_xây_dựng": "xây_dựng_năm", "xây_dựng_vào_năm": "xây_dựng_năm",
    "năm_thành_lập": "thành_lập_năm",
    "trùng_tu": "trùng_tu_năm", "năm_trùng_tu": "trùng_tu_năm",
    "tôn_tạo": "trùng_tu_năm", "tu_sửa": "trùng_tu_năm", "phục_dựng": "trùng_tu_năm",
    "người_xây_dựng": "xây_dựng_bởi", "do_xây_dựng": "xây_dựng_bởi",
    "được_xây_bởi": "xây_dựng_bởi", "kiến_trúc_sư": "thiết_kế_bởi",
    "quận": "thuộc_quận", "thuộc_quận_huyện": "thuộc_quận",
    "nằm_tại": "tọa_lạc", "vị_trí": "tọa_lạc", "địa_chỉ": "tọa_lạc",
    "nằm_ở": "tọa_lạc", "thuộc_địa_phận": "tọa_lạc",
    "thời_kỳ": "triều_đại", "thuộc_triều_đại": "triều_đại",
    "thời_đại": "triều_đại", "thuộc_triều": "triều_đại",
    "phong_cách": "phong_cách_kiến_trúc", "kiến_trúc": "phong_cách_kiến_trúc",
    "loại_hình": "phong_cách_kiến_trúc", "kiểu_kiến_trúc": "phong_cách_kiến_trúc",
    "tên_gọi_khác": "tên_khác", "còn_gọi_là": "tên_khác",
    "biệt_danh": "tên_khác", "tên_cũ": "tên_khác", "tên_gọi": "tên_khác",
    "di_sản": "công_nhận", "unesco": "công_nhận",
    "di_tích": "xếp_hạng", "xếp_hạng_di_tích": "xếp_hạng",
    "thuộc_tôn_giáo": "tôn_giáo",
    "có_liên_quan": "liên_quan_đến", "liên_quan": "liên_quan_đến",
    "gắn_liền_với": "liên_quan_đến",
    "thờ_phụng": "thờ", "tôn_thờ": "thờ", "thờ_cúng": "thờ",
}

# ---- Entity normalization ----
ENTITY_NORMALIZE = {
    "Văn Miếu": "Văn Miếu – Quốc Tử Giám",
    "Quốc Tử Giám": "Văn Miếu – Quốc Tử Giám",
    "Văn Miếu - Quốc Tử Giám": "Văn Miếu – Quốc Tử Giám",
    "nhà Lý": "Nhà Lý", "triều Lý": "Nhà Lý", "Lý triều": "Nhà Lý",
    "nhà Trần": "Nhà Trần", "triều Trần": "Nhà Trần",
    "nhà Lê": "Nhà Lê", "triều Lê": "Nhà Lê", "nhà Lê sơ": "Nhà Lê sơ",
    "nhà Nguyễn": "Nhà Nguyễn", "triều Nguyễn": "Nhà Nguyễn",
    "Pháp thuộc": "Thời Pháp thuộc", "thời Pháp": "Thời Pháp thuộc",
    "Hà nội": "Hà Nội",
}

# ---- Functions ----
def normalize_relation(rel):
    rel = rel.strip().lower().replace(" ", "_").replace("-", "_")
    rel = re.sub(r'_+', '_', rel).strip('_')
    return RELATION_NORMALIZE.get(rel, rel)

def normalize_entity(name):
    name = re.sub(r'^["\']+|["\']+$', '', name.strip())
    return ENTITY_NORMALIZE.get(name, name)

def validate_triplet(t):
    h = t.get("head", "").strip()
    r = t.get("relation", "").strip()
    tl = t.get("tail", "").strip()
    if not h or not r or not tl: return False
    if len(h) < 2 or len(tl) < 1: return False
    if len(h) > 100 or len(tl) > 300: return False
    if h == tl: return False
    if tl in ("0", "không", "không rõ", "N/A", "null", "None"): return False
    return True

def normalize_triplets(triplets):
    result, seen = [], set()
    for t in triplets:
        if not validate_triplet(t): continue
        nt = {
            "head": normalize_entity(t["head"]),
            "relation": normalize_relation(t["relation"]),
            "tail": normalize_entity(t["tail"]),
        }
        key = (nt["head"], nt["relation"], nt["tail"])
        if key not in seen:
            seen.add(key)
            result.append(nt)
    return result

def parse_json_response(response):
    # Direct parse
    try:
        data = json.loads(response)
        if isinstance(data, list): return data
    except: pass
    # Find array
    start, end = response.find('['), response.rfind(']')
    if start != -1 and end > start:
        try: return json.loads(response[start:end+1])
        except: pass
    # Fix trailing comma
    cleaned = re.sub(r',\s*\]', ']', response)
    start, end = cleaned.find('['), cleaned.rfind(']')
    if start != -1 and end > start:
        try: return json.loads(cleaned[start:end+1])
        except: pass
    # Extract individual objects
    objects = []
    for m in re.finditer(r'\{[^{}]+\}', response):
        try:
            obj = json.loads(m.group())
            if all(k in obj for k in ("head","relation","tail")):
                objects.append(obj)
        except: continue
    return objects

# ---- Prompt ----
EXTRACT_PROMPT = """Bạn là chuyên gia trích xuất thông tin lịch sử. Đọc đoạn văn bản về một di tích/danh lam ở Hà Nội và trích xuất TẤT CẢ các bộ ba (head, relation, tail).

QUY TẮC:
1. Trả về CHỈ JSON array, không có text nào khác
2. Mỗi phần tử: {{"head": "...", "relation": "...", "tail": "..."}}
3. Head: tên di tích hoặc thực thể chính
4. Tail: giá trị cụ thể (năm, tên người, quận, phong cách...)
5. Relation: dùng snake_case, ưu tiên danh sách chuẩn
6. Mỗi fact một triplet riêng

RELATION CHUẨN:
xây_dựng_năm, trùng_tu_năm, xây_dựng_bởi, triều_đại, thuộc_quận, tọa_lạc,
phong_cách_kiến_trúc, tôn_giáo, đặc_điểm, tên_khác, sự_kiện, công_nhận,
xếp_hạng, liên_quan_đến, thuộc_quần_thể, thờ, tưởng_niệm

VÍ DỤ:
Văn bản: "Chùa Một Cột hay còn gọi là Diên Hựu Tự, được vua Lý Thái Tông cho xây dựng năm 1049, tọa lạc tại quận Ba Đình. Chùa mang phong cách kiến trúc Phật giáo."
JSON:
[
  {{"head": "Chùa Một Cột", "relation": "tên_khác", "tail": "Diên Hựu Tự"}},
  {{"head": "Chùa Một Cột", "relation": "xây_dựng_bởi", "tail": "Lý Thái Tông"}},
  {{"head": "Chùa Một Cột", "relation": "xây_dựng_năm", "tail": "1049"}},
  {{"head": "Chùa Một Cột", "relation": "thuộc_quận", "tail": "Quận Ba Đình"}},
  {{"head": "Chùa Một Cột", "relation": "phong_cách_kiến_trúc", "tail": "Kiến trúc Phật giáo"}}
]

Văn bản:
{text}

JSON:"""

print("✓ Functions và prompt đã sẵn sàng")

✓ Functions và prompt đã sẵn sàng


## Bước 2b: Load Qwen & Extract triplets

**Cần GPU.** Qwen3-8B bfloat16 ≈ 16GB VRAM (vừa T4).
Nếu OOM → đổi `MODEL_NAME = "Qwen/Qwen3-4B"`

**Thời gian:** ~1-2 phút/bài × số bài

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-8B"   # ← đổi thành Qwen/Qwen3-4B nếu OOM
MAX_TEXT_LEN = 2500

input_files = sorted(Path(WIKI_RAW).glob("*.json"))
existing = set(f.stem for f in Path(OUT_TRIPLETS).glob("*.json"))
todo = [f for f in input_files if f.stem not in existing]

print(f"Model:    {MODEL_NAME}")
print(f"Input:    {len(input_files)} files")
print(f"Đã xong: {len(existing)}")
print(f"Còn lại: {len(todo)}")

if todo:
    print(f"\nĐang load model...")
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16,
        device_map="auto", trust_remote_code=True)
    model.eval()
    print("✓ Model loaded")
else:
    print("\n✓ Tất cả đã xử lý, skip!")

Model:    Qwen/Qwen3-8B
Input:    91 files
Đã xong: 0
Còn lại: 91

Đang load model...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✓ Model loaded


In [7]:
# ---- Extract loop ----
if todo:
    total_triplets = 0
    for i, f in enumerate(todo):
        rec = json.loads(f.read_text(encoding="utf-8"))
        title = rec.get("title", f.stem)
        text = rec.get("text", "")[:MAX_TEXT_LEN]
        if len(text) < 100: continue

        prompt = EXTRACT_PROMPT.format(text=text)
        messages = [{"role": "user", "content": prompt}]
        input_text = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,enable_thinking=False)
        inputs = tok(input_text, return_tensors="pt").to(model.device)

        try:
            with torch.no_grad():
                out_ids = model.generate(
                    **inputs, max_new_tokens=2048,
                    temperature=0.1, do_sample=True, top_p=0.9)
            response = tok.decode(
                out_ids[0][inputs.input_ids.shape[1]:],
                skip_special_tokens=True).strip()
        except Exception as e:
            print(f"  [{i+1}/{len(todo)}] {title}: LỖI - {e}")
            continue

        raw = parse_json_response(response)
        triplets = normalize_triplets(raw)

        result = {
            "title": title, "slug": rec.get("slug", f.stem),
            "source": rec.get("source", ""),
            "raw_triplet_count": len(raw),
            "triplet_count": len(triplets),
            "triplets": triplets,
            "raw_response": response,
        }
        Path(f"{OUT_TRIPLETS}/{f.stem}.json").write_text(
            json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
        total_triplets += len(triplets)

        if (i+1) % 1 == 0 or i == 0 or i == len(todo)-1:
            print(f"  [{i+1}/{len(todo)}] {title}: {len(raw)} raw → {len(triplets)} norm")

    print(f"\n✓ Done. Tổng: {total_triplets} triplets")
    print(f"  Output: {OUT_TRIPLETS}")
else:
    print("Đã xử lý ở lần chạy trước, skip.")

  [1/91] Bảo tàng Dân tộc học Việt Nam: 40 raw → 40 norm
  [2/91] Bảo tàng Hà Nội: 27 raw → 27 norm
  [3/91] Bảo tàng Hồ Chí Minh: 43 raw → 39 norm
  [4/91] Bảo tàng Lịch sử Quân sự Việt Nam: 18 raw → 18 norm
  [5/91] Bảo tàng Lịch sử quốc gia (Việt Nam): 30 raw → 23 norm
  [6/91] Bảo tàng Mỹ thuật Việt Nam: 41 raw → 41 norm
  [7/91] Bảo tàng Phụ nữ Việt Nam: 26 raw → 26 norm
  [8/91] Chùa Bà Đá: 43 raw → 40 norm
  [9/91] Chùa Báo Ân: 23 raw → 23 norm
  [10/91] Chùa Bối Khê: 26 raw → 26 norm
  [11/91] Chùa Bộc: 42 raw → 40 norm
  [12/91] Chùa Hòe Nhai: 41 raw → 41 norm
  [13/91] Chùa Hương: 29 raw → 29 norm
  [14/91] Chùa Kim Liên: 27 raw → 27 norm
  [15/91] Chùa Láng: 26 raw → 26 norm
  [16/91] Chùa Một Cột: 30 raw → 30 norm
  [17/91] Chùa Ngũ Xã: 24 raw → 24 norm
  [18/91] Chùa Pháp Hoa (Hà Nội): 11 raw → 11 norm
  [19/91] Chùa Phúc Khánh: 26 raw → 26 norm
  [20/91] Chùa Quán Sứ: 36 raw → 36 norm
  [21/91] Chùa Thiên Niên: 22 raw → 20 norm
  [22/91] Chùa Thầy: 30 raw → 30 norm
  [23/

In [14]:
# ---- Post-process: normalize lại + lọc noise ----
RELATION_NORMALIZE_V2 = {
    # Map về relation chuẩn
    "thiết_kế": "thiết_kế_bởi",
    "thuộc": "tọa_lạc",
    "thuộc_phường": "tọa_lạc",
    "thuộc_huyện": "thuộc_quận",
    "thuộc_thôn": "tọa_lạc",
    "thuộc_tỉnh": "tọa_lạc",
    "thuộc_phố": "tọa_lạc",
    "thuộc_phủ": "tọa_lạc",
    "thuộc_tổng": "tọa_lạc",
    "được_gọi": "tên_khác",
    "gọi_là": "tên_khác",
    "tên_gọi_trước_kia": "tên_khác",
    "tên_chính_thức": "tên_khác",
    "tên_chữ": "tên_khác",
    "tên_gốc": "tên_khác",
    "tên_pháp": "tên_khác",
    "tên_tiếng_pháp": "tên_khác",
    "tên_gọi_trước": "tên_khác",
    "tên_gọi_sau": "tên_khác",
    "tên_gọi_trong_dân_gian": "tên_khác",
    "tên_có_từ": "tên_khác",
    "nghĩa_tên": "tên_khác",
    "danh_xưng": "tên_khác",
    "khánh_thành": "xây_dựng_năm",
    "khởi_công_năm": "xây_dựng_năm",
    "khởi_công_xây_dựng": "xây_dựng_năm",
    "xây_lại_năm": "trùng_tu_năm",
    "sửa_chữa": "trùng_tu_năm",
    "dựng_năm": "xây_dựng_năm",
    "đúc_năm": "xây_dựng_năm",
    "tạc_năm": "xây_dựng_năm",
    "thời_kì": "triều_đại",
    "thời_đại_liên_quan": "triều_đại",
    "thời_điểm": "sự_kiện",
    "thời_gian": "sự_kiện",
    "diễn_ra": "sự_kiện",
    "truyền_thuyết_liên_quan": "sự_kiện",
    "lịch_sử": "sự_kiện",
    "trạng_thái": "đặc_điểm",
    "cấu_trúc": "đặc_điểm",
    "cấu_tạo": "đặc_điểm",
    "kết_cấu": "đặc_điểm",
    "bố_cục": "đặc_điểm",
    "phần_đặc_trưng": "đặc_điểm",
    "hướng": "đặc_điểm",
    "màu_sắc": "đặc_điểm",
    "chất_liệu": "đặc_điểm",
    "cảnh_quan": "đặc_điểm",
    "phế_tích": "đặc_điểm",
    "chức_năng": "đặc_điểm",
    "trưng_bày": "đặc_điểm",
    "hiện_vật": "đặc_điểm",
    "cách": "gần",
    "cách_trung_tâm": "gần",
    "tiếp_giáp": "gần",
    "tiếp_giáp_với": "gần",
    "nằm_trên": "tọa_lạc",
    "nằm_trong": "tọa_lạc",
    "phái": "tôn_giáo",
    "thuộc_hệ_phái": "tôn_giáo",
    "thuộc_phái": "tôn_giáo",
    "con": "liên_quan_đến",
    "vợ": "liên_quan_đến",
    "trị_vì": "liên_quan_đến",
    "chủ_quan_liên_quan": "liên_quan_đến",
    "công_trình_liên_quan": "liên_quan_đến",
    "nhà_sáng_tác": "xây_dựng_bởi",
    "dự_án_bởi": "xây_dựng_bởi",
    "thành_lập_bởi": "xây_dựng_bởi",
    "chủ_đầu_tư": "xây_dựng_bởi",
}

# Relations giữ nguyên (hữu ích, thêm vào standard)
KEEP_RELATIONS = {
    "trụ_trì", "chiều_dài", "chiều_cao", "diện_tích",
    "số_gian", "số_tầng", "độ_cao", "độ_dài",
}

# Minimum frequency — bỏ relation chỉ xuất hiện 1-2 lần
MIN_FREQ = 2

# Đếm frequency
from collections import Counter
all_rels = Counter()
for f in Path(OUT_TRIPLETS).glob("*.json"):
    data = json.loads(f.read_text(encoding="utf-8"))
    for t in data.get("triplets", []):
        all_rels[t["relation"]] += 1

# Re-normalize
fixed_count = 0
for f in sorted(Path(OUT_TRIPLETS).glob("*.json")):
    data = json.loads(f.read_text(encoding="utf-8"))
    new_triplets = []
    for t in data.get("triplets", []):
        rel = t["relation"]

        # Map về relation chuẩn
        if rel in RELATION_NORMALIZE_V2:
            t["relation"] = RELATION_NORMALIZE_V2[rel]
            fixed_count += 1

        # Bỏ noise (frequency thấp + không trong keep list + không chuẩn)
        if (all_rels[rel] < MIN_FREQ
            and rel not in KEEP_RELATIONS
            and rel not in STANDARD_RELATIONS
            and rel not in RELATION_NORMALIZE_V2):
            continue

        new_triplets.append(t)

    data["triplets"] = new_triplets
    data["triplet_count"] = len(new_triplets)
    f.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"✓ Fixed {fixed_count} relations")
print(f"  Chạy lại cell thống kê để kiểm tra")

✓ Fixed 265 relations
  Chạy lại cell thống kê để kiểm tra


In [18]:
# ---- Round 2: Clean 76 relations còn lại ----
RELATION_NORMALIZE_V3 = {
    # Vị trí / kết nối
    "giao_với": "gần", "giao_cắt": "gần", "cạnh": "gần",
    "kết_nối": "gần", "vị_trí_kết_nối": "gần",
    "có_phố_rẽ_vào": "gần", "đầu_kia_có_phố": "gần",
    "trực_thuộc": "tọa_lạc", "địa_điểm": "tọa_lạc",

    # Tên gọi
    "có_tên_riêng": "tên_khác", "hiệu": "tên_khác",
    "thay_đổi_tên": "tên_khác",

    # Thời gian / sự kiện
    "ngày": "sự_kiện", "quyết_định": "sự_kiện",
    "di_dời": "sự_kiện", "trở_thành": "sự_kiện",
    "được_chọn": "sự_kiện", "được_đề_xuất": "sự_kiện",

    # Đặc điểm kiến trúc
    "số_gian": "đặc_điểm", "số_cột": "đặc_điểm",
    "số_lượng_tầng": "đặc_điểm", "tượng": "đặc_điểm",
    "tượng_trưng": "đặc_điểm", "trang_trí": "đặc_điểm",
    "thượng_điện": "đặc_điểm", "tiền_đường": "đặc_điểm",
    "tầng_ba": "đặc_điểm", "tầng_hai": "đặc_điểm",
    "tầng_đế": "đặc_điểm", "cột_hiên": "đặc_điểm",
    "số_biệt_thự": "đặc_điểm",

    # Kích thước
    "chiều_dài": "diện_tích", "chiều_dài_cầu_chính": "diện_tích",
    "chiều_dài_cầu_dẫn_đường_sắt": "diện_tích",
    "chu_vi": "diện_tích", "độ_dài": "diện_tích",
    "tải_trọng": "đặc_điểm", "độ_cao": "chiều_cao",
    "cột_đường_kính": "đặc_điểm",

    # Người
    "trụ_trì": "liên_quan_đến",
    "sinh_năm": "liên_quan_đến",
    "sinh_con": "liên_quan_đến",

    # Bảo tàng / tham quan
    "loại_hiện_vật_trưng_bày": "đặc_điểm",
    "số_lượng_hiện_vật": "đặc_điểm",
    "trưng_bày_thường_xuyên": "đặc_điểm",
    "không_gian_khám_phá": "đặc_điểm",
    "lưu_ý_tham_quan": "đặc_điểm",
    "mở_cửa": "đặc_điểm",
    "miễn_phí_vé": "đặc_điểm",
    "giới_thiệu": "đặc_điểm",
    "sưu_tầm": "đặc_điểm",

    # Chức năng
    "hoạt_động": "đặc_điểm",
    "hoạt_động_liên_quan": "đặc_điểm",
    "phục_vụ": "đặc_điểm",
    "dành_cho": "đặc_điểm",
    "chứa": "đặc_điểm",
    "sản_phẩm": "đặc_điểm",
    "cây_trồng": "đặc_điểm",
}

# Relations noise — bỏ luôn
DROP_RELATIONS = {
    "xuất_ghi", "đình_miếu", "nói", "bị", "có", "từ",
    "trước_đây", "trước_đây_được", "ban_tiền",
    "hóa_thân_chuyển_thế", "đưa_con", "đến_viếng",
    "được_thăm", "đường_dẫn", "đường_ray_tàu",
    "lịch_sử_hà_nội", "ý_nghĩa",
}

fixed, dropped = 0, 0
for f in sorted(Path(OUT_TRIPLETS).glob("*.json")):
    data = json.loads(f.read_text(encoding="utf-8"))
    new_triplets = []
    for t in data["triplets"]:
        rel = t["relation"]
        if rel in DROP_RELATIONS:
            dropped += 1
            continue
        if rel in RELATION_NORMALIZE_V3:
            t["relation"] = RELATION_NORMALIZE_V3[rel]
            fixed += 1
        new_triplets.append(t)

    data["triplets"] = new_triplets
    data["triplet_count"] = len(new_triplets)
    f.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"✓ Fixed: {fixed}  |  Dropped: {dropped}")
print(f"  Chạy lại cell thống kê để kiểm tra")

✓ Fixed: 211  |  Dropped: 68
  Chạy lại cell thống kê để kiểm tra


## Bước 2.5a: Normalize food triplets

In [19]:
for sub in ["restaurants", "dishes", "districts"]:
    sub_in = Path(FOOD_RAW) / sub
    sub_out = Path(OUT_FOOD) / sub
    sub_out.mkdir(parents=True, exist_ok=True)
    if not sub_in.exists(): continue
    for f in sorted(sub_in.glob("*.json")):
        data = json.loads(f.read_text(encoding="utf-8"))
        for t in data.get("triplets", []):
            t["head"] = normalize_entity(t["head"])
            t["relation"] = normalize_relation(t["relation"])
            t["tail"] = normalize_entity(t["tail"])
        (sub_out / f.name).write_text(
            json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    n = len(list(sub_out.glob("*.json")))
    print(f"  {sub}: {n} files ✓")

  restaurants: 20 files ✓
  dishes: 17 files ✓
  districts: 16 files ✓


## Bước 2.5b: Gộp triplets → Markdown

Tạo 1 file `.md` per entity. **Review và sửa trước khi chạy bước 3.**

In [20]:
entity_data = defaultdict(lambda: {"triplets": [], "type": "Entity", "source": ""})

# Landmarks
for f in sorted(Path(OUT_TRIPLETS).glob("*.json")):
    data = json.loads(f.read_text(encoding="utf-8"))
    for t in data.get("triplets", []):
        h = t["head"]
        entity_data[h]["triplets"].append(t)
        entity_data[h]["type"] = "Landmark"
        entity_data[h]["source"] = data.get("source", "")

# Food
for sub, etype in [("restaurants","Restaurant"), ("dishes","Dish"), ("districts","Location")]:
    sub_dir = Path(OUT_FOOD) / sub
    if not sub_dir.exists(): continue
    for f in sorted(sub_dir.glob("*.json")):
        data = json.loads(f.read_text(encoding="utf-8"))
        for t in data.get("triplets", []):
            h = t["head"]
            entity_data[h]["triplets"].append(t)
            entity_data[h]["type"] = data.get("type", etype)

# Write Markdown
type_to_folder = {
    "Landmark": "landmarks", "Restaurant": "food",
    "Dish": "dishes", "Location": "locations", "Entity": "other",
}
counts = defaultdict(int)

for entity, info in sorted(entity_data.items()):
    triplets = info["triplets"]
    etype = info["type"]
    folder = type_to_folder.get(etype, "other")
    if etype == "Entity" and len(triplets) < 2: continue

    out_sub = Path(f"{OUT_MD}/{folder}")
    out_sub.mkdir(parents=True, exist_ok=True)
    slug = re.sub(r"[^\w\-]", "_", entity)
    if not slug: continue

    aliases = [t["tail"] for t in triplets if t["relation"] == "tên_khác"]
    key_attrs = {}
    for t in triplets:
        if t["relation"] in ("xây_dựng_năm", "thuộc_quận", "tọa_lạc",
                              "triều_đại", "phong_cách_kiến_trúc",
                              "loại_món", "giá_trung_bình"):
            key_attrs[t["relation"]] = t["tail"]

    lines = ["---", f"name: {entity}"]
    if aliases:
        lines.append(f"aliases: {json.dumps(aliases, ensure_ascii=False)}")
    lines.append(f"type: {etype}")
    for k, v in key_attrs.items():
        lines.append(f"{k}: {v}")
    if info.get("source"):
        lines.append(f"source: {info['source']}")
    lines.extend(["---", "", "## Relations"])

    seen = set()
    for t in triplets:
        key = (t["head"], t["relation"], t["tail"])
        if key not in seen:
            lines.append(f"- ({t['head']}, {t['relation']}, {t['tail']})")
            seen.add(key)
    lines.append("")

    (out_sub / f"{slug}.md").write_text("\n".join(lines), encoding="utf-8")
    counts[folder] += 1

print("Markdown files:")
for folder, count in sorted(counts.items()):
    print(f"  {folder}/: {count}")
print(f"  Tổng: {sum(counts.values())}")
print(f"\n→ Review kg_markdown/ trước khi chạy bước 3!")

Markdown files:
  dishes/: 17
  food/: 19
  landmarks/: 167
  locations/: 16
  Tổng: 219

→ Review kg_markdown/ trước khi chạy bước 3!


## Thống kê & Kiểm tra chất lượng

In [21]:
STANDARD_RELATIONS = {
    "xây_dựng_năm", "trùng_tu_năm", "phá_hủy_năm", "thành_lập_năm",
    "xây_dựng_bởi", "thiết_kế_bởi", "trùng_tu_bởi",
    "triều_đại", "thuộc_quận", "tọa_lạc", "thành_phố",
    "phong_cách_kiến_trúc", "tôn_giáo", "đặc_điểm", "diện_tích", "chiều_cao",
    "tên_khác", "tên_tiếng_anh", "sự_kiện", "công_nhận", "xếp_hạng",
    "liên_quan_đến", "gần", "thuộc_quần_thể", "thờ", "tưởng_niệm",
    "loại_món", "giá_trung_bình", "giờ_mở_cửa", "nổi_tiếng_vì",
    "khoảng_cách", "mô_tả", "nguồn_gốc", "đặc_trưng", "thuộc_thành_phố",
    "địa_chỉ",
}

rel_counts = defaultdict(int)
ent_counts = defaultdict(int)
total_raw, total_norm = 0, 0

for f in Path(OUT_TRIPLETS).glob("*.json"):
    data = json.loads(f.read_text(encoding="utf-8"))
    total_raw += data.get("raw_triplet_count", 0)
    for t in data.get("triplets", []):
        rel_counts[t["relation"]] += 1
        ent_counts[t["head"]] += 1
        total_norm += 1

print(f"Landmarks: {total_raw} raw → {total_norm} normalized")
if total_raw: print(f"Tỉ lệ giữ: {total_norm/total_raw:.1%}")

print(f"\nTop 10 relations:")
for rel, c in sorted(rel_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {rel}: {c}")

print(f"\nTop 10 entities:")
for ent, c in sorted(ent_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {ent}: {c} triplets")

non_std = set(rel_counts.keys()) - STANDARD_RELATIONS
if non_std:
    print(f"\n⚠ Relations không chuẩn ({len(non_std)}):")
    for r in sorted(non_std):
        print(f"  {r}: {rel_counts[r]}")
    print("→ Thêm vào RELATION_NORMALIZE rồi chạy lại extract")
else:
    print("\n✓ Tất cả relations đều chuẩn")

Landmarks: 3174 raw → 2366 normalized
Tỉ lệ giữ: 74.5%

Top 10 relations:
  đặc_điểm: 621
  liên_quan_đến: 223
  tọa_lạc: 212
  sự_kiện: 207
  tên_khác: 146
  trùng_tu_năm: 144
  phong_cách_kiến_trúc: 135
  xây_dựng_năm: 119
  thuộc_quần_thể: 99
  thờ: 74

Top 10 entities:
  Đình So: 64 triplets
  Nhà thờ Lớn Hà Nội: 51 triplets
  Đền Quán Thánh: 45 triplets
  Nhà hát Lớn Hà Nội: 44 triplets
  Phố Hàng Bè: 41 triplets
  Chùa Bộc: 40 triplets
  Chùa Tây Phương: 38 triplets
  Gò Đống Đa: 38 triplets
  Bảo tàng Mỹ thuật Việt Nam: 38 triplets
  Phố Hàng Bạc: 37 triplets

⚠ Relations không chuẩn (1):
  cắt_và_dẫn_qua: 2
→ Thêm vào RELATION_NORMALIZE rồi chạy lại extract


## Download kết quả

In [23]:
import shutil
shutil.make_archive(f"/kaggle/working/kg_data", "zip", "/kaggle/working", "kg_markdown")
print(f"✓ /kaggle/working/kg_data.zip")
# from IPython.display import FileLink
# FileLink("/kaggle/working/kg_data.zip")

✓ /kaggle/working/kg_data.zip
